# CET433 Project 2: Explainable Network IDS for SOC Analysts

## Required project work only

**Customer:** Company SOC or Managed Security Service Provider (MSSP).  
**Problem:** Detect malicious network flows and provide understandable reasons for alerts.  
**Dataset:** UNSW-NB15 official training and testing CSV files.

This notebook focuses on the required Project 2 work: binary and multiclass IDS, preprocessing without leakage, baseline and tuned model comparison, per-class recall, SHAP explanations, robustness/distribution-shift evaluation, and saved model output.

**Important:** The instructor also asks for a SOC alerts dashboard and a random-vs-temporal split comparison. The supplied UNSW-NB15 partition files do not provide a usable timestamp column, so the official held-out test comparison below is a distribution-shift check, not a true temporal split. Do not describe it as temporal validation. A timestamped dataset or a separate dashboard app is still needed if these are required as separate deliverables.


## 1. Setup

In [ ]:
import glob, json, os, platform, sys, time, warnings
from importlib.metadata import version
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Use a Python 3.12 kernel with the project dependencies installed.")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib, shap
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, precision_score,
                             recall_score, f1_score, roc_auc_score, classification_report)
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

In [ ]:
# Run from the project directory or set CET433_PROJECT_ROOT explicitly (e.g. in Colab).
start = Path(os.environ.get("CET433_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
PROJECT_DIR = next((p for p in (start, *start.parents) if (p / "model_artifact.py").is_file()), None)
if PROJECT_DIR is None:
    raise RuntimeError("Open the notebook from the project folder or set CET433_PROJECT_ROOT.")
sys.path.insert(0, str(PROJECT_DIR))
from ids_utils import log_clip, clean, prepare_features, read_flow_csv
from model_artifact import ARTIFACT_VERSION, MODEL_PACKAGES, MODEL_PATH


## 2. Load data
Needs `UNSW_NB15_training-set.csv` and `UNSW_NB15_testing-set.csv` (the partitioned files). The next cell finds them automatically:

- **Kaggle:** click **Add Input**, search for **unsw-nb15**, add the dataset, then run all cells.
- **Colab:** run the cell and upload both files when the upload button appears (or put them in the Files panel first).
- **Local:** place both files next to the notebook.


In [ ]:
TRAIN_NAME, TEST_NAME = "UNSW_NB15_training-set.csv", "UNSW_NB15_testing-set.csv"
SEARCH_ROOTS = [str(PROJECT_DIR), ".", "/content", "/kaggle/input", "/kaggle/working"]

def find_file(name):
    # looks in the working folder, Colab's /content and any Kaggle input dataset folder
    for root in SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits:
            return hits[0]
    return None

TRAIN_PATH, TEST_PATH = find_file(TRAIN_NAME), find_file(TEST_NAME)
if TRAIN_PATH is None or TEST_PATH is None:
    try:
        from google.colab import files
        print("Upload UNSW_NB15_training-set.csv and UNSW_NB15_testing-set.csv")
        files.upload()
        TRAIN_PATH, TEST_PATH = find_file(TRAIN_NAME), find_file(TEST_NAME)
    except ImportError:
        pass
if TRAIN_PATH is None or TEST_PATH is None:
    raise FileNotFoundError("Could not find the two UNSW-NB15 CSV files. On Kaggle use Add Input and search for 'unsw-nb15'; on Colab upload them; locally put them next to the notebook.")
print("Using:", TRAIN_PATH, "|", TEST_PATH)

df_raw = read_flow_csv(TRAIN_PATH)
print("Training file shape:", df_raw.shape)
df_raw.head()


## 3. Exploratory data analysis

In [ ]:
print(df_raw.dtypes.value_counts(), "\n")
print("Duplicate rows (all columns except id):", df_raw.drop(columns=["id"], errors="ignore").duplicated().sum())
print("Missing values:", int(df_raw.isna().sum().sum()))
num_only = df_raw.select_dtypes(include=np.number)
print("Inf values:", int(np.isinf(num_only).sum().sum()))
df_raw.describe().T.head(15)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df_raw["label"].map({0: "Normal", 1: "Attack"}).value_counts().plot.bar(ax=ax[0], color=["#4C78A8", "#E45756"])
ax[0].set_title("Binary class balance")
df_raw["attack_cat"].value_counts().plot.barh(ax=ax[1])
ax[1].set_title("Attack category counts"); ax[1].invert_yaxis()
plt.tight_layout(); plt.show()
print(df_raw["attack_cat"].value_counts())

## 4. Cleaning

- Drop `id` because it is only a row counter.
- Replace positive/negative `inf` with `NaN`; impute missing numeric values inside each model pipeline using training-fold statistics only.
- Drop exact duplicate rows before splitting.
- Keep `label` and `attack_cat` in the cleaned dataset as targets, but exclude them from the feature matrix.
- Apply the same cleaning to the official test set and keep it separate from training and validation.


In [ ]:
df = clean(df_raw)
df_test = clean(read_flow_csv(TEST_PATH))
print(f"Training rows: {len(df_raw)} -> {len(df)} after removing duplicates | official test rows: {len(df_test)}")

TARGETS = ["label", "attack_cat"]
X = df.drop(columns=TARGETS)
y_bin, y_multi = df["label"], df["attack_cat"]
print("Features:", X.shape[1])

## 5. Split BEFORE any fitting
Stratified on the attack category so both the binary and the multiclass task keep their class proportions in train and validation. Fixed seed.

In [ ]:
X_train, X_val, yb_train, yb_val, ym_train, ym_val = train_test_split(
    X, y_bin, y_multi, test_size=0.2, stratify=y_multi, random_state=SEED)
X_test = prepare_features(df_test, list(X.columns))
yb_test, ym_test = df_test["label"].reset_index(drop=True), df_test["attack_cat"].reset_index(drop=True)
print("Train:", X_train.shape, " Validation:", X_val.shape, " Unseen test (not used yet):", X_test.shape)
print("Attack share  train/val:", round(yb_train.mean(), 4), "/", round(yb_val.mean(), 4))

## 6. Preprocessing pipeline
One `Pipeline` per model. Everything that learns from data (median imputer, scaler, one-hot categories) is fitted **only on the training fold** because it lives inside the pipeline.

- Numeric: median impute -> `log1p` (heavy-tailed byte/packet/load features) -> standard scale
- Categorical (`proto`, `service`, `state`): one-hot, rare categories grouped, unseen categories ignored
- Class imbalance: `class_weight="balanced"` inside each classifier (training data only; SMOTE is a possible extension with an `imblearn` pipeline)

In [ ]:
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("log", FunctionTransformer(log_clip, feature_names_out="one-to-one")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.005,
                          sparse_output=False), cat_cols),
])
def make_pipe(clf):
    return Pipeline([("pre", clone(preprocess)), ("clf", clf)])

def rates(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return dict(precision=precision_score(y_true, y_pred, zero_division=0),
                recall=recall_score(y_true, y_pred), f1=f1_score(y_true, y_pred),
                fpr=float(fp / (fp + tn)), TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp))
print("numeric:", len(num_cols), "| categorical:", cat_cols)

## 7. Leakage check (documented)
1. Features contain no target or identifier columns.
2. No row index and no identical feature row appears in both train and validation.
3. The scaler saw exactly the training rows (not validation).
4. Shuffled-label test: if labels are shuffled, ROC-AUC must fall to about 0.5. If it does not, something leaks.

In [ ]:
# 1) forbidden columns
assert not {"label", "attack_cat", "id"} & set(X.columns), "target/ID column in features"

# 2) overlap
assert len(set(X_train.index) & set(X_val.index)) == 0, "index overlap"
h_tr = set(pd.util.hash_pandas_object(X_train, index=False))
h_va = set(pd.util.hash_pandas_object(X_val, index=False))
print("Identical feature rows shared by train and validation:", len(h_tr & h_va))

# 3) preprocessing fitted on train only
probe = make_pipe(DecisionTreeClassifier(random_state=SEED)).fit(X_train, yb_train)
n_seen = probe.named_steps["pre"].named_transformers_["num"].named_steps["sc"].n_samples_seen_
assert n_seen == len(X_train), "scaler saw more than the training rows"
print("Scaler fitted on", n_seen, "rows = len(X_train) OK")

# 4) shuffled-label sanity test (ROC-AUC, because F1 is inflated by the ~64% attack share)
rng = np.random.RandomState(SEED)
yb_shuffled = pd.Series(rng.permutation(yb_train.values), index=yb_train.index)
def val_auc(y_fit):
    m = make_pipe(DecisionTreeClassifier(max_depth=10, random_state=SEED)).fit(X_train, y_fit)
    return roc_auc_score(yb_val, m.predict_proba(X_val)[:, 1])
auc_shuf, auc_real = val_auc(yb_shuffled), val_auc(yb_train)
print(f"Validation ROC-AUC with shuffled labels: {auc_shuf:.3f} (expected ~0.5)  |  with real labels: {auc_real:.3f}")
assert auc_shuf < 0.6, "possible leakage: shuffled labels still score well"
assert auc_real - auc_shuf > 0.2, "real labels should beat shuffled labels clearly"
print("Leakage check passed")

## 8. Baseline models: binary task (Normal vs Attack)
Metrics reported: confusion matrix, precision, recall, F1 and False Positive Rate. **Accuracy alone is not used**: a SOC cares about missed attacks (recall) and about alert fatigue (false positives).

In [ ]:
def eval_binary(name, pipe):
    pipe.fit(X_train, yb_train)
    pred = pipe.predict(X_val)
    tn, fp, fn, tp = confusion_matrix(yb_val, pred).ravel()
    res = dict(model=name,
               precision=precision_score(yb_val, pred),
               recall=recall_score(yb_val, pred),
               f1=f1_score(yb_val, pred),
               fpr=fp / (fp + tn), TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp))
    ConfusionMatrixDisplay(confusion_matrix(yb_val, pred), display_labels=["Normal", "Attack"]).plot(cmap="Blues")
    plt.title(f"{name}: binary confusion matrix"); plt.show()
    return res

baselines = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
}
bin_results = [eval_binary(n, make_pipe(m)) for n, m in baselines.items()]
bin_table = pd.DataFrame(bin_results).set_index("model")
bin_table.round(4)

In [ ]:
target_recall, target_fpr = 0.95, 0.01
for name, r in bin_table.iterrows():
    ok = r["recall"] >= target_recall and r["fpr"] < target_fpr
    print(f"{name:20s} recall={r['recall']:.3f} fpr={r['fpr']:.3f} -> success metric {'MET' if ok else 'NOT met (notebook 2 aims to improve it)'}")

## 9. Baseline model: multiclass task (attack family)
Per-class recall shows which attack families the baseline misses; rare classes (Worms, Shellcode, Backdoor, Analysis) are the usual weak spots and the focus of the final models.

In [ ]:
multi_pipe = make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED))
multi_pipe.fit(X_train, ym_train)
pm = multi_pipe.predict(X_val)

print(classification_report(ym_val, pm, digits=3, zero_division=0))
labels = sorted(ym_val.unique())
cm = confusion_matrix(ym_val, pm, labels=labels, normalize="true")
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Logistic Regression baseline: row-normalised confusion matrix (diagonal = per-class recall)")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.show()

per_class_recall = pd.Series(recall_score(ym_val, pm, labels=labels, average=None, zero_division=0), index=labels)
per_class_recall.round(3)

## 10. Baselines with latency and ROC-AUC
The same two baselines, now with ROC-AUC and latency per prediction, so they can be compared with the tuned models in one table.

In [ ]:
baseline_pipes = {
    "Logistic Regression (baseline)": make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
    "KNN k=5 (baseline)": make_pipe(KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
}
def latency_ms(pipe, X_ref, n=200):
    # time to score ONE flow at a time, which is what a live alert stream needs
    rows = [X_ref.iloc[[i]] for i in range(min(n, len(X_ref)))]
    pipe.predict_proba(rows[0])  # warm-up
    t0 = time.perf_counter()
    for r in rows:
        pipe.predict_proba(r)
    return (time.perf_counter() - t0) / len(rows) * 1000

results = {}
fitted = {}
for name, pipe in baseline_pipes.items():
    pipe.fit(X_train, yb_train)
    proba = pipe.predict_proba(X_val)[:, 1]
    r = rates(yb_val, (proba >= 0.5).astype(int))
    r.update(roc_auc=roc_auc_score(yb_val, proba), latency_ms=latency_ms(pipe, X_val))
    results[name], fitted[name] = r, pipe
pd.DataFrame(results).T[["precision", "recall", "f1", "fpr", "roc_auc", "latency_ms"]].round(4)

## 11. Tuned models with 5-fold cross-validation (training split only)
`RandomizedSearchCV` with `StratifiedKFold(5)` optimises F1 on the training split. To keep Colab fast, the search runs on a stratified sample of up to 30,000 training rows; the best settings are then **refitted on the full training split**. The validation split is never seen by the search.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
N_TUNE = min(30000, len(X_train))
tune_idx = (X_train.assign(_y=yb_train).groupby("_y", group_keys=False)
            .apply(lambda g: g.sample(frac=N_TUNE / len(X_train), random_state=SEED)).index)
X_tune, y_tune = X_train.loc[tune_idx], yb_train.loc[tune_idx]
print("tuning rows:", len(X_tune))

search_space = {
    "Random Forest": (RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=SEED),
        {"clf__n_estimators": [100, 200], "clf__max_depth": [None, 15, 25],
         "clf__min_samples_leaf": [1, 2, 5]}),
    "LightGBM": (LGBMClassifier(class_weight="balanced", random_state=SEED, verbose=-1, n_jobs=-1),
        {"clf__n_estimators": [200, 400], "clf__learning_rate": [0.05, 0.1],
         "clf__num_leaves": [31, 63], "clf__min_child_samples": [20, 50]}),
    "MLP": (MLPClassifier(early_stopping=True, max_iter=100, random_state=SEED),
        {"clf__hidden_layer_sizes": [(64,), (128, 64)], "clf__alpha": [1e-4, 1e-3],
         "clf__learning_rate_init": [1e-3, 3e-3]}),
}
best_params, cv_scores = {}, {}
for name, (clf, grid) in search_space.items():
    s = RandomizedSearchCV(make_pipe(clf), grid, n_iter=6, cv=cv, scoring="f1",
                           random_state=SEED, n_jobs=1, refit=False)
    s.fit(X_tune, y_tune)
    best_params[name], cv_scores[name] = s.best_params_, s.best_score_
    print(f"{name:14s} best CV F1 = {s.best_score_:.4f}  {s.best_params_}")

## 12. Final fit and model comparison (validation split)

In [ ]:
tuned = {}
for name, (clf, _) in search_space.items():
    pipe = make_pipe(clone(clf)).set_params(**best_params[name])
    pipe.fit(X_train, yb_train)
    tuned[name] = pipe
    proba = pipe.predict_proba(X_val)[:, 1]
    r = rates(yb_val, (proba >= 0.5).astype(int))
    r.update(roc_auc=roc_auc_score(yb_val, proba), latency_ms=latency_ms(pipe, X_val), cv_f1=cv_scores[name])
    results[name], fitted[name] = r, pipe

comp = pd.DataFrame(results).T[["precision", "recall", "f1", "fpr", "roc_auc", "latency_ms", "cv_f1"]].astype(float)
best_base_f1 = comp.loc[[n for n in comp.index if "baseline" in n], "f1"].max()
comp["f1_gain_vs_best_baseline"] = comp["f1"] - best_base_f1
comp.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
comp["f1"].plot.barh(ax=ax[0], color="#4C78A8"); ax[0].set_title("F1 on validation (threshold 0.5)")
comp["latency_ms"].plot.barh(ax=ax[1], color="#E45756"); ax[1].set_title("Latency per prediction (ms, one flow at a time)")
plt.tight_layout(); plt.show()

### Deployment model
The product's core feature is the explanation behind each alert, so the deployed model is the **best tree-based model by validation F1** (Random Forest or LightGBM). `TreeExplainer` gives exact SHAP values quickly. The MLP stays in the comparison table; neural SHAP is slower and approximate.

In [ ]:
tree_names = ["Random Forest", "LightGBM"]
final_name = comp.loc[tree_names, "f1"].idxmax()
final_pipe = tuned[final_name]
proba_val = final_pipe.predict_proba(X_val)[:, 1]
print("Deployment model:", final_name)

## 13. Threshold from business cost and the FPR target
The threshold is the cheapest one (in analyst time) among those that keep the false positive rate under 1%, so the technical target and the business cost agree.

**Assumptions (edit them for your own case, and state them in the report):**
- A false alarm costs one analyst triage: `COST_FP_MIN` minutes.
- A missed attack costs incident handling later: `COST_FN_MIN` minutes of analyst-equivalent time.
- Business metric: analyst hours saved on the validation flows versus the best baseline at its default threshold, and scaled per 100,000 flows.

In [ ]:
COST_FP_MIN = 5
COST_FN_MIN = 120

def total_cost_min(fp, fn):
    return fp * COST_FP_MIN + fn * COST_FN_MIN

grid = np.round(np.arange(0.02, 0.99, 0.01), 2)
rows = []
for t in grid:
    r = rates(yb_val, (proba_val >= t).astype(int))
    r["threshold"] = t
    r["cost_min"] = total_cost_min(r["FP"], r["FN"])
    rows.append(r)
curve = pd.DataFrame(rows)

thr_cost_only = float(curve.loc[curve["cost_min"].idxmin(), "threshold"])
# success metric first (FPR < 1%), then the cheapest threshold among those that respect it
feasible = curve[curve["fpr"] < 0.01]
thr_cost = float(feasible.loc[feasible["cost_min"].idxmin(), "threshold"]) if len(feasible) else thr_cost_only
ok = curve[(curve["recall"] >= 0.95)]
thr_recall = float(ok.loc[ok["fpr"].idxmin(), "threshold"]) if len(ok) else None
print("Cheapest threshold ignoring the FPR target:", thr_cost_only)
print("Cheapest threshold with FPR < 1%:", thr_cost if len(feasible) else "none feasible, using the cheapest overall")
print("Threshold giving recall >= 95% with lowest FPR:", thr_recall)

plt.figure(figsize=(7, 4))
plt.plot(curve["threshold"], curve["cost_min"] / 60)
plt.axvline(thr_cost, ls="--", c="C3", label=f"chosen (FPR<1%) = {thr_cost}")
plt.xlabel("Alert threshold"); plt.ylabel("Analyst hours (validation set)"); plt.legend()
plt.title("Cost curve"); plt.show()

In [ ]:
THRESHOLD = thr_cost   # cheapest threshold that keeps FPR < 1%; use thr_recall instead if recall >= 95% is the priority
final = rates(yb_val, (proba_val >= THRESHOLD).astype(int))
print({k: round(v, 4) if isinstance(v, float) else v for k, v in final.items()})
print("Success metric (recall >= 95% and FPR < 1%):",
      "MET" if final["recall"] >= 0.95 and final["fpr"] < 0.01 else "NOT met at this threshold")

base_name = max([n for n in results if "baseline" in n], key=lambda n: results[n]["f1"])
base_cost = total_cost_min(results[base_name]["FP"], results[base_name]["FN"])
new_cost = total_cost_min(final["FP"], final["FN"])
hours_saved = (base_cost - new_cost) / 60
per_100k = hours_saved / len(X_val) * 100000
print(f"Analyst hours saved vs {base_name}: {hours_saved:.1f} h on {len(X_val)} flows (about {per_100k:.0f} h per 100,000 flows)")

## 14. Explainability with SHAP
Global view first (which features drive alerts), then three individual alerts: a true positive, a **false negative** and a **false positive**, as required.

In [ ]:
pre, clf = final_pipe.named_steps["pre"], final_pipe.named_steps["clf"]
feat_names = list(pre.get_feature_names_out())
explainer = shap.TreeExplainer(clf)

def shap_attack(Xt):
    # normalise the different SHAP output shapes to values for the "attack" class
    sv = explainer.shap_values(Xt)
    if isinstance(sv, list):
        sv = sv[1]
    sv = np.asarray(sv)
    return sv[:, :, 1] if sv.ndim == 3 else sv

def base_attack():
    ev = np.atleast_1d(explainer.expected_value)
    return float(ev[1] if len(ev) > 1 else ev[0])

n_sample = min(2000, len(X_val))
samp = X_val.sample(n_sample, random_state=SEED)
Xt_samp = pre.transform(samp)
sv_samp = shap_attack(Xt_samp)
shap.summary_plot(sv_samp, Xt_samp, feature_names=feat_names, max_display=15, show=False)
plt.title(f"SHAP summary: what pushes a flow towards 'attack' ({final_name})")
plt.tight_layout(); plt.show()

In [ ]:
pred_val = (proba_val >= THRESHOLD).astype(int)

def pick(mask, ascending):
    idx = np.where(mask)[0]
    if len(idx) == 0:
        return None
    return int(idx[np.argsort(proba_val[idx])[0 if ascending else -1]])

cases = {
    "False negative (attack missed)": pick((yb_val.values == 1) & (pred_val == 0), ascending=True),
    "False positive (false alarm)": pick((yb_val.values == 0) & (pred_val == 1), ascending=False),
}
if cases["False negative (attack missed)"] is None:   # no misses at this threshold: show the attack the model was least sure about
    cases["False negative (attack missed)"] = pick(yb_val.values == 1, ascending=True)
    print("No false negatives at this threshold; showing the attack with the lowest score instead.")
if cases["False positive (false alarm)"] is None:
    cases["False positive (false alarm)"] = pick(yb_val.values == 0, ascending=False)
    print("No false positives at this threshold; showing the normal flow with the highest score instead.")

def top_reasons(i, k=6):
    xt = pre.transform(X_val.iloc[[i]])
    sv = shap_attack(xt)[0]
    order = np.argsort(-np.abs(sv))[:k]
    return pd.DataFrame({"feature": [feat_names[j] for j in order],
                         "value_after_preprocessing": np.round(xt[0, order], 2),
                         "shap": np.round(sv[order], 3),
                         "pushes_towards": np.where(sv[order] > 0, "attack", "normal")}), xt, sv

for title, i in cases.items():
    tbl, xt, sv = top_reasons(i)
    print(f"\n=== {title}: flow #{i} | true class: {ym_val.iloc[i]} | model score: {proba_val[i]:.3f} | threshold: {THRESHOLD} ===")
    display(tbl)
    shap.plots.waterfall(shap.Explanation(values=sv, base_values=base_attack(), data=xt[0],
                                          feature_names=feat_names), max_display=10, show=False)
    plt.title(title); plt.tight_layout(); plt.show()

**How to read these for the report:** a false negative is an attack whose strongest features looked normal to the model (explain which features pulled the score down and why that attack type resembles normal traffic). A false positive is normal traffic that looked like an attack on its strongest features (for example unusual packet counts or rates). Write one short paragraph per case, using the table above.

## 15. Multiclass view (attack family)
LightGBM with the tuned settings versus the Logistic Regression baseline, compared by macro-F1 and per-class recall.

In [ ]:
lgb_params = {k: v for k, v in best_params["LightGBM"].items()}
multi_lgbm = make_pipe(LGBMClassifier(class_weight="balanced", random_state=SEED, verbose=-1, n_jobs=-1)).set_params(**lgb_params)
multi_lgbm.fit(X_train, ym_train)
multi_lr = make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)).fit(X_train, ym_train)

labels = sorted(ym_val.unique())
rec = pd.DataFrame({
    "LogReg baseline": recall_score(ym_val, multi_lr.predict(X_val), labels=labels, average=None, zero_division=0),
    "LightGBM": recall_score(ym_val, multi_lgbm.predict(X_val), labels=labels, average=None, zero_division=0)}, index=labels)
print("macro-F1  LogReg:", round(f1_score(ym_val, multi_lr.predict(X_val), average="macro"), 4),
      "| LightGBM:", round(f1_score(ym_val, multi_lgbm.predict(X_val), average="macro"), 4))
rec.round(3)

## 16. Robustness: distribution shift on the official testing set

The official testing set is kept separate from training and validation and is used only for final unseen-data evaluation. Compare validation results with official-test results to check performance under a different traffic mix.

**Limitation:** The supplied UNSW-NB15 partition CSVs do not contain a usable timestamp column. Therefore, this is an official-partition distribution-shift test, not a true temporal split. The Project 2 brief asks for a random-vs-temporal split comparison; to complete that exact item, use a timestamped dataset or obtain a valid time field and document the split method.


In [ ]:
proba_test = final_pipe.predict_proba(X_test)[:, 1]
test_res = rates(yb_test, (proba_test >= THRESHOLD).astype(int))
shift = pd.DataFrame({"validation (random split)": final, "unseen test (official testing set)": test_res}).T
shift["roc_auc"] = [roc_auc_score(yb_val, proba_val), roc_auc_score(yb_test, proba_test)]
display(shift[["precision", "recall", "f1", "fpr", "roc_auc"]].round(4))

pt = multi_lgbm.predict(X_test)
test_recall = pd.Series(recall_score(ym_test, pt, labels=labels, average=None, zero_division=0), index=labels)
print("Per-class recall on unseen test:")
display(test_recall.round(3).to_frame("recall"))

# does a threshold chosen on validation still make sense under shift?
fpr_test_curve = [rates(yb_test, (proba_test >= t).astype(int)) for t in grid]
plt.figure(figsize=(6, 4))
plt.plot(grid, [r["recall"] for r in fpr_test_curve], label="recall (test)")
plt.plot(grid, [r["fpr"] for r in fpr_test_curve], label="FPR (test)")
plt.axvline(THRESHOLD, ls="--", c="gray"); plt.xlabel("Threshold"); plt.legend(); plt.title("Threshold under shift"); plt.show()

## 17. Save the model and a demo sample

In [ ]:
MODEL_PATH.parent.mkdir(exist_ok=True)
sample_dir = PROJECT_DIR / "data_sample"
sample_dir.mkdir(exist_ok=True)
joblib.dump({"pipeline": final_pipe, "threshold": THRESHOLD, "feature_cols": list(X.columns),
             "model_name": final_name, "metadata": {
                 "artifact_version": ARTIFACT_VERSION,
                 "python": platform.python_version(),
                 "packages": {name: version(name) for name in MODEL_PACKAGES},
             }}, MODEL_PATH)

demo = df_test.sample(min(3000, len(df_test)), random_state=SEED)   # unseen by the model
demo.to_csv(sample_dir / "unseen_flows_sample.csv", index=False)

summary = {
    "deployment_model": final_name, "threshold": THRESHOLD,
    "cost_assumptions_min": {"false_alarm": COST_FP_MIN, "missed_attack": COST_FN_MIN},
    "comparison": comp.round(4).reset_index().rename(columns={"index": "model"}).to_dict(orient="records"),
    "validation": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in final.items()},
    "unseen_test": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in test_res.items()},
    "analyst_hours_saved_validation": round(hours_saved, 2),
    "analyst_hours_saved_per_100k_flows": round(per_100k, 1),
    "multiclass_recall_validation": rec.round(4).to_dict(),
    "multiclass_recall_unseen_test": test_recall.round(4).to_dict(),
}
with (PROJECT_DIR / "results.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("Saved models/ids_model.joblib, data_sample/unseen_flows_sample.csv, results.json")

## 18. Required results to report

Fill in these points from the executed results:

- Best tuned model and its improvement over the baseline.
- Precision, recall, F1, confusion matrix, and latency.
- Per-class recall for the multiclass task.
- Whether the stated business target was achieved; do not claim success if the measured results do not meet it.
- Analyst-time business metric and its assumptions.
- SHAP explanation for one false negative and one false positive.
- Difference between validation and official unseen-test results, with the temporal-split limitation stated clearly.


**Dashboard:** The companion Streamlit app is provided as `app/soc_dashboard.py` in the project bundle. Run it after generating `models/ids_model.joblib`.

**Temporal split:** The supplied UNSW-NB15 partition CSVs do not expose a reliable timestamp column. A real chronological comparison cannot be computed from these files without another timestamped source. `temporal_split_helper.py` is provided to perform the split once a real timestamp field is available; it intentionally does not fabricate dates.
